# Model Engineering : Prédiction du défaut de paiement

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

RANDOM_STATE = 42
mlflow.set_tracking_uri('file:../mlruns')


## 1. Chargement des données déjà prétraitées

In [2]:
X = np.load('../data/X_scaled.npy')
y = np.load('../data/y.npy')

print('X shape :', X.shape)
print('y shape :', y.shape)
print('\nDistribution de la cible :')
print(pd.Series(y).value_counts())
print('\nProportions :')
print(pd.Series(y).value_counts(normalize=True))


X shape : (10000, 4)
y shape : (10000,)

Distribution de la cible :
0    8149
1    1851
Name: count, dtype: int64

Proportions :
0    0.8149
1    0.1851
Name: proportion, dtype: float64


## 2. Split train / test

On conserve `stratify=y` pour préserver le déséquilibre éventuel des classes.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print('Train :', X_train.shape, y_train.shape)
print('Test  :', X_test.shape, y_test.shape)


Train : (8000, 4) (8000,)
Test  : (2000, 4) (2000,)


## 3. Définition des modèles

Les 3 modèles demandés sont :
- Régression logistique
- Arbre de décision
- Random Forest

In [4]:
models = {
    'logistic_regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'decision_tree': DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

list(models.keys())


['logistic_regression', 'decision_tree', 'random_forest']

## 4. Fonction d'évaluation

On ne se limite pas à l'accuracy, car pour un problème de défaut bancaire, `recall`, `f1`, `roc_auc` et `pr_auc` sont essentiels.

In [5]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    y_proba = None
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0)
    }

    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
        metrics['pr_auc'] = average_precision_score(y_test, y_proba)

    return metrics, y_pred, y_proba


## 5. Entraînement et comparaison des modèles

In [ ]:
all_results = []
trained_models = {}
predictions = {}

for model_name, model in models.items():
    print(f'Training {model_name}...')

    mlflow.set_experiment(model_name)

    with mlflow.start_run(run_name=f'{model_name}_baseline'):
        metrics, y_pred, y_proba = evaluate_model(model, X_train, y_train, X_test, y_test)

        # Logging paramètres
        mlflow.log_param('random_state', RANDOM_STATE)
        for param_name, param_value in model.get_params().items():
            mlflow.log_param(param_name, param_value)

        # Logging métriques
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        # Logging modèle
        mlflow.sklearn.log_model(model, artifact_path='model')

        row = {'model': model_name}
        row.update(metrics)
        all_results.append(row)

        trained_models[model_name] = model
        predictions[model_name] = {
            'y_pred': y_pred,
            'y_proba': y_proba
        }

results_df = pd.DataFrame(all_results).sort_values(by='roc_auc', ascending=False)
results_df


2026/04/03 01:58:37 INFO mlflow.tracking.fluent: Experiment with name 'logistic_regression' does not exist. Creating a new experiment.


Training logistic_regression...


2026/04/03 01:58:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 01:58:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/03 01:59:04 INFO mlflow.tracking.fluent: Experiment with name 'decision_tree' does not exist. Creating a new experiment.


Training decision_tree...


2026/04/03 01:59:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 01:59:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/03 01:59:27 INFO mlflow.tracking.fluent: Experiment with name 'random_forest' does not exist. Creating a new experiment.


Training random_forest...


2026/04/03 01:59:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 01:59:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## 6. Matrices de confusion et rapports de classification

In [ ]:
for model_name in trained_models.keys():
    print('=' * 70)
    print(f'MODEL : {model_name}')
    print('=' * 70)
    
    y_pred = predictions[model_name]['y_pred']
    
    print('Confusion matrix :')
    print(confusion_matrix(y_test, y_pred))
    
    print('\nClassification report :')
    print(classification_report(y_test, y_pred, zero_division=0))
    print('\n')


## 7. Visualisation des matrices de confusion

In [ ]:
for model_name in trained_models.keys():
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions[model_name]['y_pred'],
        ax=ax
    )
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.show()


## 8. Courbes ROC

In [ ]:
for model_name, model in trained_models.items():
    fig, ax = plt.subplots(figsize=(6, 4))
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax)
    ax.set_title(f'ROC Curve - {model_name}')
    plt.show()


## 9. Courbes Precision-Recall

In [ ]:
for model_name, model in trained_models.items():
    fig, ax = plt.subplots(figsize=(6, 4))
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax)
    ax.set_title(f'Precision-Recall Curve - {model_name}')
    plt.show()


## 10. Sélection et sauvegarde du meilleur modèle

Par défaut, on choisit le meilleur modèle selon `roc_auc`.

In [ ]:
best_model_name = results_df.iloc[0]['model']
print('Meilleur modèle sélectionné selon roc_auc :', best_model_name)
results_df